# Private CayleyPy Results Ingest npm gate
CPU-only exact-stack Task 8 load/recovery static and unit verification. No deployment or external load.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import os
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile
from pathlib import Path

WORKING = Path("/kaggle/working")
ROOT = WORKING / "cayleypy-results-ingest-gate"
PACKAGE = ROOT / "services" / "cayleypy-results-ingest"
NPM_CACHE = Path("/tmp/cayleypy-results-ingest-npm-cache")
NODE_ROOT = Path("/tmp/cayleypy-results-ingest-node")
NODE_VERSION = "v22.23.1"
NODE_ARCHIVE_NAME = "node-" + NODE_VERSION + "-linux-x64.tar.xz"
NODE_BASE_URL = "https://nodejs.org/dist/" + NODE_VERSION
PAYLOAD_B64 = '...'
EXPECTED_SHA256 = {'configs/cayleypy_results_schema_v1.json': 'bcb28d05454a8f8098c3312d53c26c2a16d4075c5c292290468cb391cd1ff662', 'configs/cayleypy_results_v1_golden.json': '6c5d241574c0ca0f859791d39ef625394f4e64d7d4e7986a04ce968f3f2ad172', 'services/cayleypy-results-ingest/package.json': '80e59683dc86dbeb7d12fb1200da52a1bfd40573c084acee55b41e61fb816929', 'services/cayleypy-results-ingest/package-lock.json': 'bf087c2dad9320ebb0091990eacded9dd751e4d3b3eb8f1c1ad8c99b12f571e3', 'services/cayleypy-results-ingest/tsconfig.json': '68f0b9d9dcfcb08ab1d2164b5463cbd670ab6f8ad62f8623a7bb7434d35ea8fa', 'services/cayleypy-results-ingest/wrangler.jsonc': '644f8b01ae9923eccf13a967ea68443b7b53bee3f0b704647dfacfe0bd57397c', 'services/cayleypy-results-ingest/vitest.config.ts': '1d6b04f23828542352038bdf1c330cc531fceefa1b227d61ae3ca0720bf3d8c0', 'services/cayleypy-results-ingest/vitest.schema.config.ts': 'b0b7f9b93338e7ed277c5be1154e0555b9186559334079415771e46c7f201285', 'services/cayleypy-results-ingest/src/mode.ts': '5053037fc5493a2b16b955b69c90bb1804cab1367bbf7b4d3e0fafb6fc3fce4d', 'services/cayleypy-results-ingest/src/github-app.ts': '0c8340ef7d64b196a98f2f275756c7b873ebfe7cc1f8362b46994774f8e4be76', 'services/cayleypy-results-ingest/src/github-writer.ts': 'e79d863dab41f89378d1887bb640b37f231f47e3d04fe6e56d3b1ccec666e322', 'services/cayleypy-results-ingest/test/github-app.test.ts': '5442e557ea7389f292551cac6cf2af4939b72305c669a881d402a6eafb1957a9', 'services/cayleypy-results-ingest/test/github-writer.test.ts': '26db2e6e6d0e0303300cd2529b6680ebe3d81825937eb6f1ce671985e8537fd5', 'services/cayleypy-results-ingest/migrations/0001_initial.sql': '8c3fe6fdc4381e123a901593962194f90aae7bfc484e6a6f5685195d05cb0ba6', 'services/cayleypy-results-ingest/migrations/0002_ingest_rate_limits.sql': '803e8b3af0dc4e1af2ddd32301a14fdba9d543cf9d5512a934f2d581f85e28f4', 'services/cayleypy-results-ingest/src/schema.ts': '66f3874041099484aec34374cccefa513c830933e5d24191681d37cc92742540', 'services/cayleypy-results-ingest/src/ids.ts': '3eafd35e8661a5b7f52651a708658f0c5665a175e249859d1dc3aa6b532a493c', 'services/cayleypy-results-ingest/src/db.ts': '7c49876d26707a134a248759e6e5f90f69abca49ba15da7c2b2aa35503dd5d9d', 'services/cayleypy-results-ingest/src/storage.ts': 'a8b01faafaaf8ada13b26cbc0b2b022432ece400511af2301ce5d8b4868c545a', 'services/cayleypy-results-ingest/src/worker.ts': 'fb16be9a94db11187c2883dac904b2fc67404ff976a321ae6d6161069d4408d6', 'services/cayleypy-results-ingest/src/replay.ts': '0acb7614048ee4f09f16a15306bb13b2ef3524abba0c98f9b8bced231f2f1d16', 'services/cayleypy-results-ingest/src/consumer.ts': '4b6c8ef7c513205568e59262049d5d7b0488b9666e02ebb46f84dc8c8ce7654c', 'services/cayleypy-results-ingest/test/schema.test.ts': 'c70556c7d2ca5d5071d009a4b9767a45fee84bb8aa78ee65d10ccc5e4851b7ba', 'services/cayleypy-results-ingest/test/apply-migrations.ts': '287f1ede2f2eaa6fa5e723a6c21dc5ad00a352ae2424663b0eb7439f7f03a24e', 'services/cayleypy-results-ingest/test/receipt.test.ts': '8d95712b00b33c5ed1d17e35679d979545bba650995711efe660fca60ad81b64', 'services/cayleypy-results-ingest/test/worker.test.ts': 'dee1a6d10906a76ff0550c3f5e08932f479e8c3922cb0ee24f2e9949da2324f7', 'services/cayleypy-results-ingest/test/consumer.test.ts': '5ffd80a1044ceb8b9216376a026fde3a38b89563a2f7b034792627234ed6e51d', 'services/cayleypy-results-ingest/test/replay.test.ts': '460bd24713a7bd180cf758e8ce61f05b5a369ae9c1e12ffa5003405a3a32a805', 'services/cayleypy-results-ingest/test/node-crypto.d.ts': '1e05f912cb3f6db3e1d1009750590d80051e4c73d5dd110213ba2845a0b57c19', 'services/cayleypy-results-ingest/test/wrangler-config.test.ts': '4feb81d2b264f27be762a8fd2413ad1f2e63778bb8309ef5f6c6647e17af8bba', 'services/cayleypy-results-ingest/load/k6-100-publishers.js': '754dfd154b554f5434815e5215c575758f605ed5285f4e374b7cc4fda45c3889', 'services/cayleypy-results-ingest/test/load-recovery-gate.test.ts': '7176b9ff0c5d7e8dbc4ddc215627857c6b336c9e0eb4f5ed3d8e8fa2aac417ae', 'services/cayleypy-results-ingest/test/recovery-audit.ts': 'c834070f09a788d343b73bb6e6a345042912604eb9f80d84a545155d3b49f58a'}

if ROOT.exists():
    shutil.rmtree(ROOT)
if NPM_CACHE.exists():
    shutil.rmtree(NPM_CACHE)
if NODE_ROOT.exists():
    shutil.rmtree(NODE_ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD_B64))) as archive:
    archive.extractall(ROOT)

shasums = urllib.request.urlopen(
    NODE_BASE_URL + "/SHASUMS256.txt", timeout=60
).read().decode("utf-8")
expected_node_sha = next(
    line.split()[0]
    for line in shasums.splitlines()
    if line.endswith("  " + NODE_ARCHIVE_NAME)
)
node_archive = urllib.request.urlopen(
    NODE_BASE_URL + "/" + NODE_ARCHIVE_NAME, timeout=180
).read()
observed_node_sha = hashlib.sha256(node_archive).hexdigest()
if observed_node_sha != expected_node_sha:
    raise RuntimeError("Node archive checksum mismatch")
NODE_ROOT.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(node_archive), mode="r:xz") as archive:
    archive.extractall(NODE_ROOT)
NODE_BIN = NODE_ROOT / NODE_ARCHIVE_NAME.removesuffix(".tar.xz") / "bin"
os.environ["PATH"] = str(NODE_BIN) + os.pathsep + os.environ["PATH"]
(WORKING / "node-bootstrap.json").write_text(
    json.dumps(
        {"version": NODE_VERSION, "archive": NODE_ARCHIVE_NAME, "sha256": observed_node_sha},
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

observed = {}
for relative in EXPECTED_SHA256:
    observed[relative] = hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
if observed != EXPECTED_SHA256:
    raise RuntimeError("embedded payload checksum mismatch")
(WORKING / "payload-sha256.json").write_text(
    json.dumps(observed, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

combined_log = WORKING / "npm-gate.log"
combined_log.write_text("", encoding="utf-8")
results = {"payload_sha256": observed, "commands": []}
command_outputs = {}
command_stdout = {}
RUN_ENV = {
    **os.environ,
    "NPM_CONFIG_CACHE": str(NPM_CACHE),
    "NPM_CONFIG_REGISTRY": "https://registry.npmjs.org/",
    "NPM_CONFIG_FETCH_RETRIES": "3",
    "NPM_CONFIG_UPDATE_NOTIFIER": "false",
    "NO_COLOR": "1",
    "FORCE_COLOR": "0",
}


def run(label: str, argv: list[str]) -> int:
    completed = subprocess.run(
        argv,
        cwd=PACKAGE,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=RUN_ENV,
    )
    command_stdout[label] = completed.stdout
    output = completed.stdout + completed.stderr
    command_outputs[label] = output
    section = f"\n===== {label} (exit={completed.returncode}) =====\n{output}"
    print(section, flush=True)
    with combined_log.open("a", encoding="utf-8") as handle:
        handle.write(section)
    (WORKING / f"{label}.log").write_text(output, encoding="utf-8")
    results["commands"].append(
        {"label": label, "argv": argv, "exit_code": completed.returncode}
    )
    return completed.returncode

REGISTRY_QUERIES = {
    "npm-view-vitest-pool": [
        "npm", "view", "@cloudflare/vitest-pool-workers@0.19.0",
        "version", "dist-tags", "dependencies", "peerDependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-wrangler": [
        "npm", "view", "wrangler@4.115.0",
        "version", "dist-tags", "dependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-workers-types": [
        "npm", "view", "@cloudflare/workers-types@5.20260729.1",
        "version", "dist-tags", "dependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-vitest": [
        "npm", "view", "vitest@4.1.10",
        "version", "dist-tags", "dependencies", "peerDependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-workerd": [
        "npm", "view", "workerd@1.20260729.1",
        "version", "dist-tags", "dependencies", "engines", "dist.integrity", "--json",
    ],
}
REGISTRY_EXPECTED = {
    "npm-view-vitest-pool": "0.19.0",
    "npm-view-wrangler": "4.115.0",
    "npm-view-workers-types": "5.20260729.1",
    "npm-view-vitest": "4.1.10",
    "npm-view-workerd": "1.20260729.1",
}

run("node-version", ["node", "--version"])
run("npm-version", ["npm", "--version"])
for label, argv in REGISTRY_QUERIES.items():
    run(label, argv)
run("npm-install-package-lock-only", ["npm", "install", "--package-lock-only", "--no-audit", "--no-fund"])
run("npm-ci", ["npm", "ci", "--no-audit", "--no-fund"])
run(
    "npm-ls-critical",
    [
        "npm", "ls", "@cloudflare/vitest-pool-workers", "@cloudflare/workers-types",
        "vitest", "wrangler", "miniflare", "workerd", "--json",
    ],
)
run("npm-test", ["npm", "test"])
run("npm-test-worker", ["npm", "exec", "--", "vitest", "run", "--config", "vitest.config.ts"])
run("npm-typecheck", ["npm", "run", "typecheck"])

registry_metadata = {}
registry_parse_errors = []
for label, expected_version in REGISTRY_EXPECTED.items():
    try:
        metadata = json.loads(command_stdout[label])
    except (KeyError, json.JSONDecodeError) as exc:
        registry_parse_errors.append({"label": label, "error": type(exc).__name__})
        continue
    registry_metadata[label] = metadata
registry_exact = not registry_parse_errors and all(
    registry_metadata[label].get("version") == expected
    for label, expected in REGISTRY_EXPECTED.items()
)
(WORKING / "registry-metadata.json").write_text(
    json.dumps(
        {
            "expected_versions": REGISTRY_EXPECTED,
            "exact": registry_exact,
            "parse_errors": registry_parse_errors,
            "registry": registry_metadata,
        },
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

lockfile = PACKAGE / "package-lock.json"
lock = json.loads(lockfile.read_text(encoding="utf-8")) if lockfile.is_file() else {"packages": {}}
lock_packages = lock.get("packages", {})
workerd_paths = sorted(
    path for path in lock_packages if path == "node_modules/workerd" or path.endswith("/node_modules/workerd")
)
workerd_versions = sorted({
    lock_packages[path].get("version") for path in workerd_paths if lock_packages[path].get("version")
})

def installed_version(relative: str) -> str | None:
    path = PACKAGE / "node_modules" / relative / "package.json"
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))["version"]

expected_stack = {
    "@cloudflare/vitest-pool-workers": "0.19.0",
    "@cloudflare/workers-types": "5.20260729.1",
    "vitest": "4.1.10",
    "wrangler": "4.115.0",
    "miniflare": "4.20260722.1",
    "workerd": "1.20260729.1",
}
resolved_stack = {name: installed_version(name) for name in expected_stack}
resolved_stack_exact = (
    resolved_stack == expected_stack
    and workerd_versions == ["1.20260729.1"]
    and bool(workerd_paths)
)
resolved_report = {
    "expected": expected_stack,
    "resolved": resolved_stack,
    "exact": resolved_stack_exact,
    "lockfile_workerd_paths": workerd_paths,
    "lockfile_workerd_versions": workerd_versions,
}
(WORKING / "resolved-stack.json").write_text(
    json.dumps(resolved_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

warning_markers = (
    "newer than the latest supported date",
    "falling back to",
    "unsupported compatibility date",
    "does not support compatibility date",
)
warning_hits = []
for label, output in command_outputs.items():
    for line_number, line in enumerate(output.splitlines(), start=1):
        lowered = line.lower()
        if any(marker in lowered for marker in warning_markers):
            warning_hits.append({"label": label, "line": line_number, "text": line})
warning_report = {
    "compatibility_date": "2026-07-28",
    "markers": list(warning_markers),
    "hits": warning_hits,
    "passed": not warning_hits,
}
(WORKING / "compatibility-warning-scan.json").write_text(
    json.dumps(warning_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

post_install_sha256 = {
    relative: hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
    for relative in EXPECTED_SHA256
}
post_install_mismatches = {
    relative: {"expected": EXPECTED_SHA256[relative], "observed": observed_sha}
    for relative, observed_sha in post_install_sha256.items()
    if observed_sha != EXPECTED_SHA256[relative]
}
(WORKING / "post-install-payload-sha256.json").write_text(
    json.dumps(post_install_sha256, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

prior_commands_passed = all(item["exit_code"] == 0 for item in results["commands"])
gate_assertions = {
    "prior_commands_passed": prior_commands_passed,
    "registry_versions_exact": registry_exact,
    "resolved_stack_exact": resolved_stack_exact,
    "workerd_override_exact": workerd_versions == ["1.20260729.1"],
    "compatibility_warning_scan_clean": not warning_hits,
    "post_install_payload_exact": not post_install_mismatches,
    "lockfile_present": lockfile.is_file(),
}
gate_exit = 0 if all(gate_assertions.values()) else 1
gate_output = json.dumps(gate_assertions, indent=2, sort_keys=True) + "\n"
command_outputs["gate-assertions"] = gate_output
(WORKING / "gate-assertions.log").write_text(gate_output, encoding="utf-8")
with combined_log.open("a", encoding="utf-8") as handle:
    handle.write(f"\n===== gate-assertions (exit={gate_exit}) =====\n{gate_output}")
results["commands"].append(
    {"label": "gate-assertions", "argv": [], "exit_code": gate_exit}
)

if lockfile.is_file():
    shutil.copy2(lockfile, WORKING / "package-lock.json")
results.update(
    {
        "lockfile_present": lockfile.is_file(),
        "registry_versions_exact": registry_exact,
        "resolved_stack": resolved_report,
        "compatibility_warning_scan": warning_report,
        "post_install_payload_sha256": post_install_sha256,
        "post_install_payload_mismatches": post_install_mismatches,
        "gate_assertions": gate_assertions,
        "all_commands_passed": all(item["exit_code"] == 0 for item in results["commands"]),
    }
)
(WORKING / "npm-gate-results.json").write_text(
    json.dumps(results, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
shutil.rmtree(ROOT)
shutil.rmtree(NPM_CACHE)
shutil.rmtree(NODE_ROOT)
print(json.dumps(results, indent=2, sort_keys=True), flush=True)
